In [36]:
from tavily import TavilyClient
from datetime import datetime, timedelta
import json
import asyncio
import json
from typing import List, Dict, Any, Optional

from pydantic import BaseModel, Field, ValidationError
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# Import shared clients
import shared_clients
from shared_clients import shared_clients


#!/usr/bin/env python3
"""
Manager Multiprocessing - Import Manager Agent and run multiprocessing analysis
"""

import asyncio
import multiprocessing as mp
from typing import List, Dict, Any
from concurrent.futures import ProcessPoolExecutor
from time import time


#!/usr/bin/env python3
"""
Simple Multiprocessing Manager - Direct multiprocessing without classes
"""

import asyncio
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor
from Manager_Agent import quick_analysis

import asyncio
import json
from typing import Dict, Any

# Import shared clients
import shared_clients
from shared_clients import shared_clients



# Section 1). Search in Internet, what is the current long/short logic 

In [37]:
# To install: pip install tavily-python
from tavily import TavilyClient

def get_company_name_by_ticker(ticker: str) -> str:
    """
    Get company name from ticker symbol using Tavily search.
    
    Args:
        ticker (str): Stock ticker symbol (e.g., "BBW", "AAPL", "TSLA")
    
    Returns:
        str: Company name
    """
    client = TavilyClient("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")
    
    # Enhanced query to get company name only
    query = f"what is the company name for stock ticker {ticker.upper()}, return only the company name"
    
    try:
        response = client.search(
            query=query,
            include_answer="advanced",
            search_depth="advanced"
        )
        
        # Extract company name from response
        if response and 'answer' in response:
            company_name = response['answer']
            print(f"✅ Found company name for {ticker}: {company_name}")
            return company_name
        else:
            print(f"❌ No company name found for ticker {ticker}")
            return f"Unknown Company ({ticker})"
            
    except Exception as e:
        print(f"❌ Error searching for {ticker}: {e}")
        return f"Error ({ticker})"

def get_stock_analysis_tavily_dual(company_name):
    """
    Get stock analysis using two Tavily searches and integrate results
    
    Args:
        company_name (str): Company name (e.g., 'Build-A-Bear Workshop', 'Apple Inc.')
    
    Returns:
        dict: Contains summary['sell_and_buy'] and summary['business_logic']
    """
    try:
        # Initialize Tavily client
        client = TavilyClient("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")
        
        print(f"🔍 Starting dual Tavily search for {company_name}...")
        
        # SEARCH 1: Sell/Buy Analysis (Market focus, validation, fundamental level)
        query1 = f"List 3–9 stock drivers (Macro/Market/Fundamentals) for {company_name}, with +/– factors tied to future growth, risks, or resolutions. Return ONLY numbered list(each line summarize as key words bullet point)."
        print(f"📊 Search 1: Sell/Buy analysis for {company_name}...")
        
        response1 = client.search(
            query=query1,
            include_answer="advanced",
            search_depth="advanced",
            topic="general",
            max_results=10
        )
        
        # SEARCH 2: Business Logic Analysis (Moat, business model)
        query2 = f"I want a very specific deep detailed explanation on the business model: moat, scale, customers, and logic for {company_name}, if no, just say some positive and negative about the company"
        
        print(f"�� Search 2: Business logic analysis for {company_name}...")
        
        response2 = client.search(
            query=query2,
            include_answer="advanced",
            search_depth="advanced",
            topic="general",
            max_results=10
        )
        
        # Extract content and answer from Search 1 (Sell/Buy)
        sell_buy_content_list = []
        if 'results' in response1:
            for result in response1['results']:
                if 'content' in result and result['content']:
                    sell_buy_content_list.append(result['content'])
        
        sell_buy_content = " | ".join(sell_buy_content_list)
        sell_buy_answer = response1.get('answer', '')
        
        # Extract content and answer from Search 2 (Business Logic)
        business_logic_content_list = []
        if 'results' in response2:
            for result in response2['results']:
                if 'content' in result and result['content']:
                    business_logic_content_list.append(result['content'])
        
        business_logic_content = " | ".join(business_logic_content_list)
        business_logic_answer = response2.get('answer', '')
        
        # Create structured summary
        summary = {
            'sell_and_buy': sell_buy_answer,  # This will now be a numbered list
            'business_logic': business_logic_answer
        }
        
        # Create result dictionary
        result = {
            'company_name': company_name,
            'summary': summary,
            'sell_and_buy_content': sell_buy_content,
            'business_logic_content': business_logic_content,
            'sell_and_buy_response_time': response1.get('response_time', 0),
            'business_logic_response_time': response2.get('response_time', 0),
            'sell_and_buy_results': len(response1.get('results', [])),
            'business_logic_results': len(response2.get('results', []))
        }
        
        print(f"✅ Dual Tavily search complete for {company_name}")
        print(f"💰 Sell/Buy results: {result['sell_and_buy_results']}")
        print(f"🏢 Business Logic results: {result['business_logic_results']}")
        print(f"⏱️ Total response time: {result['sell_and_buy_response_time'] + result['business_logic_response_time']} seconds")
        
        return result
        
    except Exception as e:
        print(f"❌ Error in dual Tavily search for {company_name}: {e}")
        return {
            'company_name': company_name,
            'summary': {
                'sell_and_buy': 'Error occurred during sell/buy search',
                'business_logic': 'Error occurred during business logic search'
            },
            'sell_and_buy_content': '',
            'business_logic_content': ''
        }

# MAIN PIPELINE FUNCTION
def get_stock_analysis_pipeline(ticker):
    """
    Complete pipeline: Ticker -> Company Name -> Two Searches
    
    Args:
        ticker (str): Stock ticker symbol (e.g., "BBW")
    
    Returns:
        dict: Complete analysis with company name and two search results
    """
    print(f"🚀 Starting analysis pipeline for ticker: {ticker}")
    
    # STEP 1: Get company name from ticker
    company_name = get_company_name_by_ticker(ticker)
    
    # STEP 2: Use company name for two searches
    analysis = get_stock_analysis_tavily_dual(company_name)
    
    # Add ticker to result
    analysis['ticker'] = ticker
    
    return analysis

# Simple usage function
def get_stock_summary_simple(ticker):
    """
    Simple function to get stock summary - just input ticker
    
    Args:
        ticker (str): Stock ticker symbol
    
    Returns:
        dict: Summary with sell_and_buy and business_logic
    """
    result = get_stock_analysis_pipeline(ticker)
    return result['summary']


In [38]:
ticker = "OPEN"
language = "English"
company_name = get_company_name_by_ticker(ticker)
print(f"Company: {company_name}")

# Test stock analysis
analysis = get_stock_summary_simple(ticker)
print(f"Sell/Buy Analysis: {analysis['sell_and_buy']}")
print(f"Business Logic: {analysis['business_logic']}")
sell_and_buy = analysis['sell_and_buy']
business_logic = analysis['business_logic']

✅ Found company name for OPEN: Opendoor Technologies Inc.
Company: Opendoor Technologies Inc.
🚀 Starting analysis pipeline for ticker: OPEN
✅ Found company name for OPEN: Opendoor Technologies Inc.
🔍 Starting dual Tavily search for Opendoor Technologies Inc....
📊 Search 1: Sell/Buy analysis for Opendoor Technologies Inc....
�� Search 2: Business logic analysis for Opendoor Technologies Inc....
✅ Dual Tavily search complete for Opendoor Technologies Inc.
💰 Sell/Buy results: 10
🏢 Business Logic results: 10
⏱️ Total response time: 2.35 seconds
Sell/Buy Analysis: 1. **Housing Market Cyclicality** – Mortgage rates near 7% and reduced transaction volumes (down 25% year-over-year) create headwinds, but potential Fed rate cuts could accelerate recovery and boost transaction activity

2. **Margin Improvement Trajectory** – Gross margins evolved from negative in 2022 to 5-6% in 2024 through enhanced pricing algorithms and shorter inventory holding periods, though margins remain fragile to pricin

## In future, Frontend need to print these two things out

In [39]:
print(f"Sell and Buy: {sell_and_buy}")
print(f"Business Logic: {business_logic}")


Sell and Buy: 1. **Housing Market Cyclicality** – Mortgage rates near 7% and reduced transaction volumes (down 25% year-over-year) create headwinds, but potential Fed rate cuts could accelerate recovery and boost transaction activity

2. **Margin Improvement Trajectory** – Gross margins evolved from negative in 2022 to 5-6% in 2024 through enhanced pricing algorithms and shorter inventory holding periods, though margins remain fragile to pricing errors

3. **Capital Intensity Risk** – iBuying model requires substantial balance sheet capacity to hold home inventory, with holding costs escalating if transaction velocity slows

4. **Strategic Platform Pivot** – Transition from product-based to platform approach with agent partnerships in 11 markets provides asset-light revenue streams and reduces inventory risk

5. **Operational Efficiency Gains** – 33% fixed cost cuts, workforce reductions, and streamlined operations position company for rapid scaling when market conditions improve

6. *

## Calling supervisor Agent, to generate sub queries for the manager agents

In [40]:
#!/usr/bin/env python3

from __future__ import annotations

from typing import List, Dict, Optional, Literal
from pydantic import BaseModel, Field, ValidationError, conint
from langchain_deepseek import ChatDeepSeek
import os

# Set API key
os.environ["DEEPSEEK_API_KEY"] = "sk-43e9043c7ab8480393d34367f2ae997e"

# -------------------------
# 1) Define schemas (Pydantic v2)
# -------------------------
Priority = conint(ge=1, le=10)  # 1..10
Impact = Literal["Positive", "Negative", "Neutral", "Mixed"]

class SubManagerQuery(BaseModel):
    manager_id: str = Field(..., description="Unique ID (A, B, C, D...)")
    query: str = Field(..., description="Specific query for this sub-manager to investigate")
    dimension: str = Field(..., description="Market dimension this query focuses on")
    priority: Priority = Field(..., description="Priority level 1-10 (1=highest priority)")
    expected_impact: Impact = Field(..., description="Expected impact type")

class SupervisorResult(BaseModel):
    total_sub_managers: conint(ge=1, le=10) = Field(..., description="Number of sub-managers (1-10)")
    sub_managers: List[SubManagerQuery] = Field(default_factory=list, description="List of sub-manager queries")
    buy_side_summary: List[str] = Field(default_factory=list, description="Summary points from buy side")
    sell_side_summary: List[str] = Field(default_factory=list, description="Summary points from sell side")
    business_logic: List[str] = Field(default_factory=list, description="Key business logic insights")
    manager_agent_query: str = Field(..., description="Rephrased and summarized query for Manager Agent")


# -------------------------
# 2) Supervisor Agent - FIXED
# -------------------------
class SupervisorAgent:
    def __init__(self):
        """Initialize the Supervisor Agent (lazy clients)."""
        self.llm: Optional[ChatDeepSeek] = None
        self.structured_llm = None

    def initialize_shared_clients(self):
        """Initialize shared clients synchronously."""
        try:
            from LLM_Call_Agent import DEEPSEEK_API_KEY
            self.llm = ChatDeepSeek(
                model="deepseek-chat",
                temperature=0.1,
                api_key=DEEPSEEK_API_KEY
            )
        except ImportError:
            self.llm = ChatDeepSeek(
                model="deepseek-chat",
                temperature=0.1,
            )

        self.structured_llm = self.llm.with_structured_output(SupervisorResult)
        print("✅ Supervisor Agent initialized with LangChain DeepSeek structured output")

    def breakdown_query(
        self,
        sell_buy_analysis: str,
        business_logic: str,
        ticker: str,
        language: str
    ) -> SupervisorResult:
        """Break down the query into sub-manager tasks and create Manager Agent query."""
        try:
            if self.structured_llm is None:
                self.initialize_shared_clients()

            print(f"🔍 Supervisor Agent analyzing {ticker} ...")

            # SIMPLIFIED PROMPT - Much cleaner and clearer
            prompt = f"""You are a Supervisor Agent analyzing {ticker} stock.

SELL/BUY ANALYSIS:
{sell_buy_analysis}

BUSINESS LOGIC:
{business_logic}

TASK: Create 1-6 sub-manager queries based on the analysis above.

REQUIREMENTS:
1. Extract key points from sell/buy analysis
2. Create specific sub-queries for investigation
3. Each query should be 30 words or less
4. Output in {language}
5. CRITICAL: Every sub-manager MUST have ALL required fields

OUTPUT FORMAT:
- total_sub_managers: number (1-10)
- sub_managers: list with ALL fields for EACH sub-manager:
  * manager_id: "A", "B", "C", "D", "E", "F" (unique letters)
  * query: specific investigation query
  * dimension: market dimension (e.g., "Financial", "Competitive", "Growth")
  * priority: number 1-10 (1=highest)
  * expected_impact: "Positive", "Negative", "Neutral", or "Mixed"
- buy_side_summary: list of positive points
- sell_side_summary: list of negative points  
- business_logic: list of business insights
- manager_agent_query: rephrased summary for Manager Agent

CRITICAL: Every sub-manager MUST have expected_impact field. No exceptions.

All text must be in {language}."""

            print("🤖 Making LLM call...")
            result: SupervisorResult = self.structured_llm.invoke(prompt)
            
            # Debug: Check if result is None
            if result is None:
                print("❌ LLM returned None - trying alternative approach...")
                # Try a simpler approach
                simple_prompt = f"Analyze {ticker} stock. Create 3 sub-queries for investigation. Output in {language}."
                result = self.structured_llm.invoke(simple_prompt)
                
                if result is None:
                    raise Exception("LLM returned None even with simple prompt - check API key and connection")
            
            print(f"✅ LLM response received: {type(result)}")
            print(f"✅ Result has total_sub_managers: {hasattr(result, 'total_sub_managers')}")

            if result.total_sub_managers != len(result.sub_managers):
                result.total_sub_managers = len(result.sub_managers)

            print("✅ Supervisor breakdown complete!")
            print(f"→ Total sub-managers: {result.total_sub_managers}")
            print(f"→ Manager Agent Query: {result.manager_agent_query}")
            return result

        except ValidationError as ve:
            print(f"❌ Validation failed: {ve}")
            print(f"   - Error details: {ve}")
            raise Exception(f"Supervisor Agent validation failed: {ve}")

        except Exception as e:
            print(f"❌ Error in supervisor breakdown: {e}")
            print(f"   - Error type: {type(e)}")
            print(f"   - Error details: {str(e)}")
            raise Exception(f"Supervisor Agent processing failed: {e}")

    def get_sub_manager_queries(self, result: SupervisorResult) -> Dict[str, str]:
        """Extract sub-manager queries as a {manager_id: query} dict."""
        return {sm.manager_id: sm.query for sm in result.sub_managers}

    def get_manager_agent_query(self, result: SupervisorResult) -> str:
        """Extract the rephrased Manager Agent query."""
        return result.manager_agent_query

    def print_breakdown_summary(self, result: SupervisorResult) -> None:
        """Pretty print a summary of the breakdown."""
        print("\n" + "=" * 60)
        print("�� SUPERVISOR SUMMARY")
        print("=" * 60)
        print(f"🎯 Sub-Managers: {result.total_sub_managers}")
        print(f"🎯 Manager Agent Query: {result.manager_agent_query}")

        print("\n📈 BUY SIDE SUMMARY:")
        for i, point in enumerate(result.buy_side_summary, 1):
            print(f"  {i}. {point}")

        print("\n📉 SELL SIDE SUMMARY:")
        for i, point in enumerate(result.sell_side_summary, 1):
            print(f"  {i}. {point}")

        print("\n🏢 BUSINESS LOGIC:")
        for i, point in enumerate(result.business_logic, 1):
            print(f"  {i}. {point}")

        print("\n🔍 SUB-MANAGER QUERIES:")
        for sm in result.sub_managers:
            print(f"  {sm.manager_id}. [{sm.priority}] {sm.dimension} — {sm.expected_impact}")
            print(f"     {sm.query}\n")


# -------------------------
# 4) Convenience functions - FIXED
# -------------------------
def create_supervisor_breakdown(
    sell_buy_analysis: str, business_logic: str, ticker: str, language: str
) -> SupervisorResult:
    try:
        sup = SupervisorAgent()
        result = sup.breakdown_query(sell_buy_analysis, business_logic, ticker, language)
        
        # Check if result is None
        if result is None:
            raise Exception("Supervisor Agent returned None - check LLM response")
        
        return result
        
    except Exception as e:
        print(f"❌ Error in create_supervisor_breakdown: {e}")
        print(f"   - sell_buy_analysis length: {len(sell_buy_analysis) if sell_buy_analysis else 0}")
        print(f"   - business_logic length: {len(business_logic) if business_logic else 0}")
        print(f"   - ticker: {ticker}")
        print(f"   - language: {language}")
        raise Exception(f"Failed to create supervisor breakdown: {e}")


def get_sub_manager_queries_simple(result: SupervisorResult) -> Dict[str, str]:
    return {sm.manager_id: sm.query for sm in result.sub_managers}


def get_manager_agent_query_simple(result: SupervisorResult) -> str:
    return result.manager_agent_query

In [41]:

internet_search_result = create_supervisor_breakdown(sell_and_buy, business_logic, ticker, language)

# Access results
print(f"Total sub-managers: {internet_search_result.total_sub_managers}")
for sub_manager in internet_search_result.sub_managers:
    print(f"Manager {sub_manager.manager_id}: {sub_manager.query}")

# Get queries as dictionary
queries = get_sub_manager_queries_simple(internet_search_result)
print(f"Queries: {queries}")

✅ Supervisor Agent initialized with LangChain DeepSeek structured output
🔍 Supervisor Agent analyzing OPEN ...
🤖 Making LLM call...
2025-09-11 22:14:11,472 - INFO - HTTP Request: POST https://api.deepseek.com/v1/chat/completions "HTTP/1.1 200 OK"
✅ LLM response received: <class '__main__.SupervisorResult'>
✅ Result has total_sub_managers: True
✅ Supervisor breakdown complete!
→ Total sub-managers: 6
→ Manager Agent Query: Analyze OPEN stock considering housing market cyclicality headwinds, margin improvement trajectory, capital intensity risks, strategic platform pivot benefits, operational efficiency gains, partnership diversification, EBITDA profitability milestone, competitive pressures, and delisting risk management through reverse stock split.
Total sub-managers: 6
Manager A: Analyze impact of potential Fed rate cuts on mortgage rates and transaction volume recovery for OPEN's iBuying business model
Manager B: Evaluate margin sustainability from 5-6% levels given pricing algorithm

### Make Sure Frontend Print this out


In [42]:
internet_search_result
print(internet_search_result.sell_side_summary)
print(internet_search_result.buy_side_summary)
print(internet_search_result.business_logic)

sell_side_summary = internet_search_result.sell_side_summary
buy_side_summary = internet_search_result.buy_side_summary
business_logic = internet_search_result.business_logic


['Mortgage rates near 7% creating transaction volume headwinds', 'Capital intensive model requires substantial balance sheet capacity', 'Intense competition pressures market share and margins', 'Delisting risk with reverse stock split may alienate investors', 'Margins remain fragile to pricing errors in volatile market']
['EBITDA turned positive demonstrating business model scalability', '33% fixed cost cuts and operational efficiency gains', 'Platform pivot reduces inventory risk with asset-light revenue', 'Partnerships with Zillow and Realtor.com create valuable deal flow', 'Margin improvement from negative to 5-6% through pricing algorithms']
['iBuying model purchases homes 3-8% below market for convenience premium', 'Revenue from 5-7% service fees plus title insurance and mortgage sales', 'Scale advantage across 50 markets enables cost-of-capital advantage', 'Requires purchasing below market to cover repairs, holding costs, and margins', 'Transitioning from product-based to platfor

# Section2). Mutliprocess To Call All Manager  + Chain of Thought AI

In [43]:
# Reload all modules to ensure latest versions
import importlib
import sys
import asyncio
from typing import List, Dict, Any

# Force clear any cached instances
modules_to_clear = [
    'Manager_Agent',
    'shared_clients',
    'Chain_of_Thought_Agent',
    'Market_Expectation_Agent',
    'Revenue_Segmentation_Read_Agent',
    'Macro_Analyst_Agent',
    'Financial_Metrics_Analyst_Agent'
]

for module in modules_to_clear:
    if module in sys.modules:
        del sys.modules[module]

# Reload all modules
modules_to_reload = [
    'Manager_Agent',
    'Chain_of_Thought_Agent',
    'Market_Expectation_Agent',
    'Revenue_Segmentation_Read_Agent',
    'Macro_Analyst_Agent',
    'Financial_Metrics_Analyst_Agent'
]

for module in modules_to_reload:
    importlib.reload(__import__(module))

# Import the functions we need
from Manager_Agent import quick_analysis, get_manager_result, get_manager_progress, get_manager_instance
from Chain_of_Thought_Agent import (
    ChainOfThoughtAgent,
    ChainOfThoughtResult,
    generate_mermaid_code,
    concurrent_call_generate_impact_chain_with_mermaid_auto
)

print("✅ All modules reloaded and imported successfully!")
print("📦 Available functions:")
print("   - quick_analysis")
print("   - get_manager_instance")
print("   - concurrent_call_generate_impact_chain_with_mermaid_auto (NOW ASYNC!)")

# Convert queries dictionary to list for proper indexing
# Add error handling for queries variable
if queries is None:
    print("❌ Error: queries is None. Please check if internet_search_result was created successfully.")
    queries = {}
elif isinstance(queries, str):
    print(f"❌ Error: queries is a string, not a dictionary: {queries}")
    queries = {}
elif not isinstance(queries, dict):
    print(f"❌ Error: queries is not a dictionary, it's a {type(queries)}: {queries}")
    queries = {}

queries_list = list(queries.values()) if queries else []
query_keys = list(queries.keys()) if queries else []

print(f"📋 Total queries: {len(queries_list)}")
print(f"🔑 Query keys: {query_keys}")
print(f"📝 Query list: {queries_list}")
print(f"🌐 Language setting: {language}")

# CRITICAL: Initialize Manager Agent for concurrent processing
print("🚀 Initializing Manager Agent for concurrent processing...")
manager = await get_manager_instance()

print("✅ Manager Agent initialized and ready for concurrent calls")

async def process_single_query_concurrent(query_index: int, language: str = "English"):
    """Process a single query with multiprocessing - NO user_query needed!"""
    
    try:
        # Get the query from the list
        user_query = queries_list[query_index]
        
        # Use the pre-initialized manager instance directly
        # Manager Agent will use its own shared clients instance
        await shared_clients.initialize()
        
        # FIXED: run_complete_analysis returns only final_results, not a tuple
        final_results = await manager.run_complete_analysis(
            user_query=user_query,
            ticker=ticker,
            shared_clients=shared_clients,
            language=language
        )
        
        # Create the same structure as quick_analysis
        manager_result = {
            "user_query": user_query,
            "ticker": ticker,
            "user_id": f"concurrent_{query_index}",
            "agent_results": final_results,
            "agents_result": final_results,  # Use final_results as agents_result too
            "execution_summary": {
                "total_agents_executed": len(final_results),
                "successful_executions": len([r for r in final_results.values() if not str(r).startswith("Error")]),
                "failed_executions": len([r for r in final_results.values() if str(r).startswith("Error")]),
                "execution_time": "N/A"
            }
        }
        
        # 🚀 ASYNC Chain of Thought with context - AUTO query assignment
        # This is the crucial part that was missing!
        print(f"🔍 Query {query_index}: Starting Chain of Thought analysis...")
        try:
            from Chain_of_Thought_Agent import concurrent_call_generate_impact_chain_with_mermaid_auto
            
            # ✅ NOW ASYNC - Chain of Thought runs concurrently!
            print(f"🔍 Query {query_index}: Calling Chain of Thought Agent...")
            chain_result = await concurrent_call_generate_impact_chain_with_mermaid_auto(
                ticker=ticker,
                verification_links=[],
                verification_reasoning="",
                agent_analysis_results=manager_result['agent_results'],
                total_queries=queries_list,  # Use the list, not the dict
                query_index=query_index,
                context={
                    "user_query": user_query,
                    "language": language,
                    "total_queries": len(queries_list),
                    "query_key": query_keys[query_index] if query_index < len(query_keys) else f"Q{query_index}"
                },
                language=language
            )
            
            # Add chain result to manager result
            manager_result['chain_of_thought'] = chain_result
            manager_result['mermaid_diagram'] = chain_result.get('mermaid_code', '')
            
            print(f"✅ Query {query_index}: Chain of Thought completed successfully!")
            print(f"📊 Query {query_index}: Chain result keys: {list(chain_result.keys())}")
            
        except Exception as e:
            print(f"❌ Query {query_index}: Chain_of_Thought_Agent failed: {e}")
            print(f"❌ Query {query_index}: Error type: {type(e)}")
            import traceback
            print(f"❌ Query {query_index}: Traceback: {traceback.format_exc()}")
            manager_result['chain_of_thought'] = None
            manager_result['mermaid_diagram'] = ''
        
        return {
            "query_index": query_index,
            "query_key": query_keys[query_index] if query_index < len(query_keys) else f"Q{query_index}",
            "status": "success",
            "result": manager_result
        }
        
    except Exception as e:
        print(f"❌ Query {query_index} failed: {e}")
        return {
            "query_index": query_index,
            "query_key": query_keys[query_index] if query_index < len(query_keys) else f"Q{query_index}",
            "status": "failed",
            "error": str(e)
        }

# Process all queries concurrently
print("🚀 Starting concurrent processing of all queries...")
tasks = [process_single_query_concurrent(i, language) for i in range(len(queries_list))]
final_results = await asyncio.gather(*tasks, return_exceptions=True)

# Process results
successful = 0
failed = 0

for i, result in enumerate(final_results):
    if isinstance(result, Exception):
        print(f"❌ Query {i} (Exception): {result}")
        failed += 1
    elif result.get("status") == "success":
        print(f"✅ Query {result['query_index']} ({result['query_key']}): Success")
        successful += 1
    else:
        print(f"❌ Query {result['query_index']} ({result['query_key']}): Failed - {result.get('error', 'Unknown error')}")
        failed += 1

print(f"\n📈 SUMMARY:")
print(f"   Total queries processed: {len(final_results)}")
print(f"   Successful: {successful}")
print(f"   Failed: {failed}")

# Show Chain of Thought results
print(f"\n🧠 CHAIN OF THOUGHT RESULTS:")
for i, result in enumerate(final_results):
    if isinstance(result, Exception):
        continue
    if result.get("status") == "success":
        chain_data = result['result'].get('chain_of_thought')
        mermaid_data = result['result'].get('mermaid_diagram')
        print(f"📊 Query {result['query_index']} ({result['query_key']}):")
        print(f"   Chain of Thought: {'✅ Present' if chain_data else '❌ Missing'}")
        print(f"   Mermaid Diagram: {'✅ Present' if mermaid_data else '❌ Missing'}")
        if chain_data:
            print(f"   Chain keys: {list(chain_data.keys()) if isinstance(chain_data, dict) else 'Not a dict'}")
        if mermaid_data:
            print(f"   Mermaid length: {len(mermaid_data)} characters")

✅ LLM_Call_Agent integration available
✅ Using API keys from LLM_Call_Agent
🤖 Shared Client Pool created (not initialized yet)
✅ LLM_Call_Agent integration available
✅ Using API keys from LLM_Call_Agent
🤖 Shared Client Pool created (not initialized yet)
✅ LLM_Call_Agent integration available
✅ Using API keys from LLM_Call_Agent
🤖 Shared Client Pool created (not initialized yet)
✅ All modules reloaded and imported successfully!
📦 Available functions:
   - quick_analysis
   - get_manager_instance
   - concurrent_call_generate_impact_chain_with_mermaid_auto (NOW ASYNC!)
📋 Total queries: 6
🔑 Query keys: ['A', 'B', 'C', 'D', 'E', 'F']
📝 Query list: ["Analyze impact of potential Fed rate cuts on mortgage rates and transaction volume recovery for OPEN's iBuying business model", 'Evaluate margin sustainability from 5-6% levels given pricing algorithm improvements and inventory holding period optimization', 'Assess capital intensity risks and balance sheet capacity requirements for inventory ho

In [44]:
final_results

[{'query_index': 0,
  'query_key': 'A',
  'status': 'success',
  'result': {'user_query': "Analyze impact of potential Fed rate cuts on mortgage rates and transaction volume recovery for OPEN's iBuying business model",
   'ticker': 'OPEN',
   'user_id': 'concurrent_0',
   'agent_results': {'Market_Expectation_Agent_Result': "✅ Using fresh data for OPEN. **SIMILAR TREND MAPPING**: \n<Similar Trend Time: uptrend3 2024-10-23, 2024-10-28>\n<Reason: because similar macro as Federal Reserve's anticipated 25 basis point rate cut on Nov 7 meeting, creating favorable monetary conditions for growth stocks. Market concentration concerns in top tech stocks (MSFT, AAPL) driving capital rotation to alternative opportunities. Increased Chinese IPO activity in US markets indicating improved cross-border investment flows., micro as Positive sentiment around tech earnings season with analysts like Cramer advising against trading big winners ahead of earnings reports. Potential beneficiary of small-cap s

In [45]:
final_results

[{'query_index': 0,
  'query_key': 'A',
  'status': 'success',
  'result': {'user_query': "Analyze impact of potential Fed rate cuts on mortgage rates and transaction volume recovery for OPEN's iBuying business model",
   'ticker': 'OPEN',
   'user_id': 'concurrent_0',
   'agent_results': {'Market_Expectation_Agent_Result': "✅ Using fresh data for OPEN. **SIMILAR TREND MAPPING**: \n<Similar Trend Time: uptrend3 2024-10-23, 2024-10-28>\n<Reason: because similar macro as Federal Reserve's anticipated 25 basis point rate cut on Nov 7 meeting, creating favorable monetary conditions for growth stocks. Market concentration concerns in top tech stocks (MSFT, AAPL) driving capital rotation to alternative opportunities. Increased Chinese IPO activity in US markets indicating improved cross-border investment flows., micro as Positive sentiment around tech earnings season with analysts like Cramer advising against trading big winners ahead of earnings reports. Potential beneficiary of small-cap s

In [46]:
# More detailed extraction with error handling
COT_prepare_Dynamic_Rating = []

for i, result in enumerate(final_results):
    if result.get('status') == 'success':
        # The chain_of_thought data is nested under result['result']['chain_of_thought']
        result_data = result.get('result', {})
        chain_data = result_data.get('chain_of_thought', {})
        
        if chain_data:
            # Extract the nested chain_of_thought data
            nested_chain = chain_data.get('chain_of_thought', {})
            
            # Ensure all required fields exist
            chain_result = {
                'ticker': ticker,  # Use the global ticker variable
                'query': chain_data.get('query', ''),
                'query_index': chain_data.get('query_index', i),
                'impact_chain': nested_chain.get('impact_chain', ''),
                'final_direction': nested_chain.get('final_direction', ''),
                'chain_explanation': nested_chain.get('chain_explanation', ''),
                'node_count': nested_chain.get('node_count', 0),
                'edge_count': nested_chain.get('edge_count', 0),
                'events': nested_chain.get('events', []),
                'mermaid_code': chain_data.get('mermaid_code', ''),
                'mermaid_diagram': result_data.get('mermaid_diagram', ''),
                'context': chain_data.get('context', {})
            }
            COT_prepare_Dynamic_Rating.append(chain_result)
            print(f"✅ Added chain {i+1}: {chain_result['final_direction']} - {chain_result['query'][:50]}...")
        else:
            print(f"⚠️ No chain_of_thought data for result {i+1}")

print(f"📊 Total chain results extracted: {len(COT_prepare_Dynamic_Rating)}")

# Display the extracted data structure
for i, chain in enumerate(COT_prepare_Dynamic_Rating):
    print(f"\n🔗 Chain {i+1}:")
    print(f"   Query: {chain['query']}")
    print(f"   Direction: {chain['final_direction']}")
    print(f"   Impact Chain: {chain['impact_chain']}")
    print(f"   Explanation: {chain['chain_explanation']}")
    print(f"   Events: {len(chain['events'])} events")
    print(f"   Mermaid Code: {chain['mermaid_code']}")
    print(f"   Mermaid Diagram: {chain['mermaid_diagram']}")

✅ Added chain 1: Short Term Up - Analyze impact of potential Fed rate cuts on mortg...
✅ Added chain 2: Long Term Up - Evaluate margin sustainability from 5-6% levels gi...
✅ Added chain 3: Short Term Down - Assess capital intensity risks and balance sheet c...
✅ Added chain 4: Long Term Up - Examine platform pivot success with agent partners...
✅ Added chain 5: Long Term Down - Investigate competitive pressure from Zillow, Redf...
✅ Added chain 6: Short Term Down - Analyze delisting risk management through reverse ...
📊 Total chain results extracted: 6

🔗 Chain 1:
   Query: Analyze impact of potential Fed rate cuts on mortgage rates and transaction volume recovery for OPEN's iBuying business model
   Direction: Short Term Up
   Impact Chain: Potential Fed rate cuts → Market data shows similar monetary easing events caused +2.88% return from 2025-05-09 to 2025-05-19 → Mortgage rates decline (-5.11% over three months) → Housing transaction volume recovery (+0.71% retail sales growth) → 

# Section 3). Conclusion AI 

In [47]:
import asyncio
import json
from typing import Dict, Any
from pydantic import BaseModel, Field
from langchain_deepseek import ChatDeepSeek
import os

# Set API key
os.environ["DEEPSEEK_API_KEY"] = "sk-43e9043c7ab8480393d34367f2ae997e"

# Define the Pydantic model
class ChainOfThoughtConclusionResult(BaseModel):
    short_term_impact: str = Field(description="Detailed analysis of short-term impact (50 words with clear safe or not safe in current position price)")
    long_term_outlook: str = Field(description="Detailed analysis of long-term outlook (50 words with clear safe or not safe in current position price)")
    Catalyst: str = Field(description="From all u have read and analyst what id the three things that chanage (in the chains), then the whole story change (50 words)")
   

class ChainOfThoughtConclusionAgent:
    def __init__(self):
        """Initialize the Chain of Thought Conclusion Agent with direct LLM initialization"""
        self.llm_agent = None
        self.structured_llm = None
        
        # Direct LLM initialization (same as Supervisor Agent)
        try:
            from LLM_Call_Agent import DEEPSEEK_API_KEY
            self.llm_agent = ChatDeepSeek(
                model="deepseek-chat",
                temperature=0.1,
                api_key=DEEPSEEK_API_KEY
            )
        except ImportError:
            self.llm_agent = ChatDeepSeek(
                model="deepseek-chat",
                temperature=0.1,
            )
        
        if self.llm_agent is None:
            raise Exception("❌ CRITICAL ERROR: LLM Call Agent is None. Check API keys and network connection.")
        
        # Create structured LLM
        self.structured_llm = self.llm_agent.with_structured_output(ChainOfThoughtConclusionResult)
        print("✅ Chain of Thought Conclusion Agent initialized with direct LLM")
    
    async def analyze_impact(self, business_logic: str, chain_of_thought: str, language: str = "English") -> Dict[str, Any]:
        """
        Analyze short-term vs long-term impact based on business logic and chain of thought
        
        Args:
            business_logic: The business logic/moat analysis
            chain_of_thought: The current sell/buy chain of thought result
            language: Language for output
            
        Returns:
            Dictionary with analysis results
        """
        
        prompt = f"""
        You are an expert financial analyst specializing in short-term vs long-term impact analysis.

        TASK: Analyze the impact on the company based on business logic (moat) and current sell/buy chain of thought.

        BUSINESS LOGIC (MOAT): {business_logic}

        CHAIN OF THOUGHT RESULT: {chain_of_thought}

        ANALYSIS FRAMEWORK:
        1. **SHORT-TERM IMPACT ANALYSIS**:
           - Identify immediate challenges, troubles, or potential bubbles
           - Assess market sentiment and short-term headwinds
           - Evaluate temporary vs structural issues
           - Consider cyclical vs secular factors

        2. **LONG-TERM OUTLOOK ANALYSIS**:
           - Evaluate company's ability to overcome short-term troubles or maintain the business sustainability (success)
           - Assess competitive advantages and moat sustainability
           - Analyze strategic positioning for long-term success
           - Consider fundamental business model strength

        3. **CATALYST IDENTIFICATION**:
           - Identify the three key things that could change the whole story
           - Focus on factors that would significantly alter the chain of thought

        LANGUAGE REQUIREMENT: All output must be in {language}. Respond entirely in {language}.

        OUTPUT REQUIREMENTS:
        - Be specific and data-driven
        - Distinguish between temporary and permanent factors
        - Provide clear investment implications
        - Use professional financial analysis language
        - Keep responses concise but comprehensive
        - Provide reward scores as decimal numbers (0.0 to 1.0)
        """
        
        try:
            # Use structured LLM invoke method
            result = self.structured_llm.invoke(prompt)
            return result.model_dump()  # Convert Pydantic model to dict (v2 syntax)
            
        except Exception as e:
            raise Exception(f"Chain of Thought Conclusion analysis failed: {e}")

# Convenience function with global language detection
async def analyze_chain_conclusion(business_logic: str, chain_of_thought: str, language: str = None) -> Dict[str, Any]:
    """
    Convenience function to analyze chain of thought conclusion
    
    Args:
        business_logic: The business logic/moat analysis
        chain_of_thought: The current sell/buy chain of thought result
        language: Language for output (if None, will use global language variable)
        
    Returns:
        Dictionary with analysis results
    """
    try:
        # Use global language variable if not provided
        if language is None:
            try:
                language = globals()['language']
                print(f"🌐 Using global language setting: {language}")
            except KeyError:
                language = "English"
                print("⚠️ Global language variable not found, using default: English")
        
        agent = ChainOfThoughtConclusionAgent()
        result = await agent.analyze_impact(business_logic, chain_of_thought, language)
        return result
    except Exception as e:
        print(f"❌ Error in analyze_chain_conclusion: {e}")
        return {
            "short_term_impact": f"Analysis failed: {e}",
            "long_term_outlook": "Analysis failed",
            "Catalyst": "Analysis failed",
        }

In [48]:
Conclusion_Result = await analyze_chain_conclusion(business_logic, final_results)

🌐 Using global language setting: English
✅ Chain of Thought Conclusion Agent initialized with direct LLM
2025-09-11 22:17:06,249 - INFO - HTTP Request: POST https://api.deepseek.com/v1/chat/completions "HTTP/1.1 200 OK"


In [49]:
Conclusion_Result

{'short_term_impact': 'NOT SAFE in current position. Short-term faces multiple headwinds: capital intensity risks during slow periods (-13% historical impact), competitive pressure causing -25.8% YoY revenue decline, and reverse stock split signaling distress (-2.145% to -6.317% historical declines). Despite strong liquidity ($2.15B working capital), negative earnings yield (-7.5%) and poor income quality (-28.4x) indicate fundamental profitability challenges outweigh temporary strengths.',
 'long_term_outlook': 'NOT SAFE for sustainable growth. Long-term structural issues persist: intense competition from Zillow/Redfin eroding market share, margin compression despite platform pivot efforts, and business model vulnerability to housing cycles. While platform transition to asset-light model and agent partnerships show promise (+4.936% historical gains), negative profitability metrics and revenue decline suggest unsustainable competitive positioning in crowded proptech space.',
 'Catalyst

# 4). Dynamic Rating AI 

In [50]:
import os
import json
from typing import Dict, Any, List
from pydantic import BaseModel, Field
from langchain_deepseek import ChatDeepSeek

# Set API key
os.environ["DEEPSEEK_API_KEY"] = "sk-43e9043c7ab8480393d34367f2ae997e"

# Define the Pydantic model for Dynamic Rating with reasoning
class DynamicRatingResult(BaseModel):
    ShortTermRisk: float = Field(description="Short-term risk score (0.0 to 1.0)")
    ShortTermReward: float = Field(description="Short-term reward score (0.0 to 1.0)")
    LongTermRisk: float = Field(description="Long-term risk score (0.0 to 1.0)")
    LongTermReward: float = Field(description="Long-term reward score (0.0 to 1.0)")
    Price_In: str = Field(description="What is the current price in? (50 words), and how much left that is not priced in?")
    Price_Out: str = Field(description="What is the current market don't price in/not discover? (50 words), once price in how much it could go up?/down?")
    Reasoning: str = Field(description="Detailed explanation of how the scores were calculated, including price pattern analysis")

def extract_current_trend_from_redis_data(redis_data: Dict[str, Any]) -> Dict[str, Any]:
    """
    Extract current trend data directly from Redis data structure
    
    Args:
        redis_data: The complete Redis data structure
        
    Returns:
        Dictionary with current trend information
    """
    
    # Extract current trends (most recent trend)
    current_trends = redis_data.get("current_trends", {})
    
    # Get the latest trend (usually the highest numbered trend)
    latest_trend_key = None
    latest_trend_data = None
    
    if current_trends:
        # Find the trend with the highest number (most recent)
        trend_keys = list(current_trends.keys())
        if trend_keys:
            # Sort by trend number to get the latest
            trend_numbers = []
            for key in trend_keys:
                try:
                    # Extract number from trend key (e.g., "uptrend13" -> 13)
                    number = int(''.join(filter(str.isdigit, key)))
                    trend_numbers.append((number, key))
                except:
                    continue
            
            if trend_numbers:
                # Get the trend with highest number
                latest_number, latest_trend_key = max(trend_numbers)
                latest_trend_data = current_trends[latest_trend_key]
    
    if not latest_trend_data:
        return {
            "error": "No current trend data available"
        }
    
    # Extract price-related information
    current_data = {
        "ticker": redis_data.get("ticker", "UNKNOWN"),
        "current_trend_period": latest_trend_data.get("current", "Unknown"),
        "trend_direction": "Up" if "uptrend" in latest_trend_key else "Down",
        "estimated_price": latest_trend_data.get("Estimate_price", 0),
        "max_return": latest_trend_data.get("Max Return", 0),
        "day_average_return": latest_trend_data.get("day average_return", 0),
        "week_average_return": latest_trend_data.get("week average return", 0),
        "slope_of_trend": latest_trend_data.get("Slope of stock trend", 0),
        "duration_days": latest_trend_data.get("How Long it Take", 0),
        "volatility": latest_trend_data.get("return rate variance", 0),
        "macro_reason": latest_trend_data.get("summary", {}).get("macro_reason", ""),
        "micro_reason": latest_trend_data.get("summary", {}).get("micro_reason", ""),
        "trend_key": latest_trend_key
    }
    
    return current_data

class DynamicRatingAgent:
    def __init__(self):
        """Initialize the Dynamic Rating Agent with direct LLM initialization"""
        self.llm_agent = None
        self.structured_llm = None
        
        # Direct LLM initialization
        try:
            from LLM_Call_Agent import DEEPSEEK_API_KEY
            self.llm_agent = ChatDeepSeek(
                model="deepseek-chat",
                temperature=0.1,
                api_key=DEEPSEEK_API_KEY
            )
        except ImportError:
            self.llm_agent = ChatDeepSeek(
                model="deepseek-chat",
                temperature=0.1,
            )
        
        if self.llm_agent is None:
            raise Exception("❌ CRITICAL ERROR: LLM Call Agent is None. Check API keys and network connection.")
        
        # Create structured LLM
        self.structured_llm = self.llm_agent.with_structured_output(DynamicRatingResult)
        print("✅ Dynamic Rating Agent initialized with direct Redis access")
    
    async def get_stock_data_from_redis(self, ticker: str) -> Optional[Dict]:
        """
        Retrieve stock trend data for a given ticker using the EXACT same method as Stock Trend Agent
        
        Args:
            ticker (str): Stock ticker symbol
            
        Returns:
            Optional[Dict]: Stock trend data or None if not found
        """
        try:
            # Use shared Redis connection exactly like Stock Trend Agent
            if 'shared_clients' in globals() and shared_clients:
                redis_client = shared_clients.get_stock_trend_redis()
                collection_name = "Stock_Trend_INFOS"
            else:
                return {"error": "No shared clients available"}
            
            # Use the EXACT same Redis key format as Stock Trend Agent
            redis_key = f"{collection_name}:{ticker.upper()}_trends"
            data_str = await redis_client.get(redis_key)
            
            if not data_str:
                return {"error": f"No data found for ticker {ticker}"}
            
            # Parse JSON data exactly like Stock Trend Agent
            stock_data = json.loads(data_str)
            
            return stock_data
            
        except Exception as e:
            print(f"❌ Error getting Redis data for {ticker}: {e}")
            return {"error": f"Failed to get Redis data: {e}"}
    
    async def generate_dynamic_rating(
        self, 
        chain_of_thought_results: List[Dict[str, Any]], 
        conclusion_result: Dict[str, Any],
        ticker: str,
        language: str = "English"
    ) -> Dict[str, Any]:
        """
        Generate dynamic rating scores with DIRECT REDIS ACCESS using Stock Trend Agent method
        
        Args:
            chain_of_thought_results: List of chain of thought results
            conclusion_result: Conclusion analysis result
            ticker: Stock ticker symbol
            language: Language for output
            
        Returns:
            Dictionary with dynamic rating scores and reasoning
        """
        
        # Get current pricing data directly from Redis using Stock Trend Agent method
        print(f"🎯 Getting current pricing data from Redis for {ticker}...")
        redis_data = await self.get_stock_data_from_redis(ticker)
        
        if "error" in redis_data:
            print(f"⚠️ Using fallback pricing analysis: {redis_data['error']}")
            current_pricing = {
                "ticker": ticker,
                "estimated_price": 0,
                "trend_direction": "Unknown",
                "max_return": 0,
                "volatility": 0,
                "macro_reason": "No current data available",
                "micro_reason": "No current data available"
            }
        else:
            # Extract current trend data from Redis
            current_pricing = extract_current_trend_from_redis_data(redis_data)
        
        # Extract key information from conclusion result
        short_term_impact = conclusion_result.get('short_term_impact', '')
        long_term_outlook = conclusion_result.get('long_term_outlook', '')
        catalyst = conclusion_result.get('Catalyst', '')
        
        # Extract chain information
        chain_summary = ""
        for i, chain in enumerate(chain_of_thought_results):
            direction = chain.get('final_direction', '')
            chain_text = chain.get('impact_chain', '')
            chain_summary += f"Chain {i+1}: {direction} - {chain_text}\n"
        
        prompt = f"""
        You are a PRICING-FOCUSED financial analyst AI. Your PRIMARY GOAL is to analyze CURRENT PRICING vs POTENTIAL MOVEMENTS.

        ASSIGN 4 SCORES (0.0 to 1.0 scale) based on CURRENT PRICING ANALYSIS:

        **CURRENT PRICING ANALYSIS FRAMEWORK:**

        1. **Short Term Risk** (price reversal risk from current level):
           • If current price already reflects good news → HIGH risk (0.6-0.8)
           • If price has surged >20% recently → MAX risk (0.8-1.0)
           • If current price is reasonable vs fundamentals → LOW risk (0.2-0.4)
           • If price already dropped on bad news → LOW risk (0.2-0.4)

        2. **Short Term Reward** (upside potential from current level):
           • If current price doesn't reflect positive catalysts → HIGH reward (0.7-1.0)
           • If price is at support levels → HIGH reward (0.6-0.8)
           • If current price already reflects upside → LOW reward (0.2-0.4)
           • If no near-term catalysts → LOW reward (0.1-0.3)

        3. **Long Term Risk** (structural issues not priced in):
           • If current price ignores long-term challenges → HIGH risk (0.6-0.8)
           • If price reflects structural concerns → LOW risk (0.2-0.4)
           • If fundamentals deteriorating vs current price → HIGH risk (0.7-1.0)

        4. **Long Term Reward** (long-term value vs current price):
           • If current price undervalues growth potential → HIGH reward (0.6-0.8)
           • If price fairly values long-term prospects → MID reward (0.4-0.6)
           • If current price overvalues long-term potential → LOW reward (0.2-0.4)

        **CURRENT PRICING DATA FROM REDIS:**
        - Ticker: {ticker}
        - Current Estimated Price: ${current_pricing.get('estimated_price', 0):.2f}
        - Trend Direction: {current_pricing.get('trend_direction', 'Unknown')}
        - Max Recent Return: {current_pricing.get('max_return', 0):.1%}
        - Daily Average Return: {current_pricing.get('day_average_return', 0):.2%}
        - Weekly Average Return: {current_pricing.get('week_average_return', 0):.2%}
        - Trend Slope: {current_pricing.get('slope_of_trend', 0):.2f}
        - Duration: {current_pricing.get('duration_days', 0)} days
        - Volatility: {current_pricing.get('volatility', 0):.3f}
        - Trend Period: {current_pricing.get('current_trend_period', 'Unknown')}
        - Trend Key: {current_pricing.get('trend_key', 'Unknown')}

        **MACRO FACTORS:**
        {current_pricing.get('macro_reason', 'N/A')[:200]}...

        **COMPANY-SPECIFIC FACTORS:**
        {current_pricing.get('micro_reason', 'N/A')[:200]}...

        **ADDITIONAL ANALYSIS DATA:**
        - Short Term Impact: {short_term_impact}
        - Long Term Outlook: {long_term_outlook}
        - Key Catalysts: {catalyst}
        - Chain Directions: {chain_summary}

        **PRICE IN vs PRICE OUT ANALYSIS:**
        1. **Price_In**: What is currently priced into the stock? What percentage of recent events/news is already reflected in current price?
        2. **Price_Out**: What is NOT priced in? What surprises could move the stock significantly? What is the market missing?

        **OUTPUT FORMAT:**
        {{
          "ShortTermRisk": <float 0.0-1.0>,
          "ShortTermReward": <float 0.0-1.0>,
          "LongTermRisk": <float 0.0-1.0>,
          "LongTermReward": <float 0.0-1.0>,
          "Price_In": "Current pricing analysis: what's priced in and what's not (50 words)",
          "Price_Out": "Market blind spots: what's not priced in and potential impact (50 words)",
          "Reasoning": "Pricing-focused reasoning: 1) Current price level analysis, 2) What's priced in vs not, 3) Risk/reward from current levels"
        }}

        **CRITICAL FOCUS:**
        - Analyze CURRENT PRICE LEVEL vs POTENTIAL MOVEMENTS
        - Determine what's PRICED IN vs NOT PRICED IN
        - Assess RISK/REWARD from CURRENT LEVELS
        - Consider MARKET EXPECTATIONS vs REALITY
        - Use the Redis current trend data to understand current pricing

        Output in language: {language}
        """
        
        try:
            result = self.structured_llm.invoke(prompt)
            return result.model_dump()  # Convert Pydantic model to dict (v2 syntax)
        except Exception as e:
            raise Exception(f"Dynamic Rating analysis failed: {e}")

# Convenience function with global language detection
async def generate_dynamic_rating_scores(
    chain_of_thought_results: List[Dict[str, Any]], 
    conclusion_result: Dict[str, Any],
    ticker: str,
    language: str = None
) -> Dict[str, Any]:
    """
    Convenience function to generate dynamic rating scores with DIRECT REDIS ACCESS
    
    Args:
        chain_of_thought_results: List of chain of thought results
        conclusion_result: Conclusion analysis result
        ticker: Stock ticker symbol
        language: Language for output (if None, will use global language variable)
        
    Returns:
        Dictionary with dynamic rating scores and reasoning
    """
    try:
        # Use global language variable if not provided
        if language is None:
            try:
                language = globals()['language']
                print(f"🌐 Using global language setting: {language}")
            except KeyError:
                language = "English"
                print("⚠️ Global language variable not found, using default: English")
        
        agent = DynamicRatingAgent()
        result = await agent.generate_dynamic_rating(chain_of_thought_results, conclusion_result, ticker, language)
        return result
    except Exception as e:
        print(f"❌ Error in generate_dynamic_rating_scores: {e}")
        return {
            "ShortTermRisk": 0.5,
            "ShortTermReward": 0.5,
            "LongTermRisk": 0.5,
            "LongTermReward": 0.5,
            "Price_In": f"Error occurred: {e}",
            "Price_Out": f"Error occurred: {e}",
            "Reasoning": f"Error occurred: {e}"
        }

In [51]:
dynamic_Rating =  await generate_dynamic_rating_scores(COT_prepare_Dynamic_Rating, Conclusion_Result, ticker, language)



✅ Dynamic Rating Agent initialized with direct Redis access
🎯 Getting current pricing data from Redis for OPEN...
2025-09-11 22:17:24,220 - INFO - HTTP Request: POST https://api.deepseek.com/v1/chat/completions "HTTP/1.1 200 OK"


### Paramter Summarize from the whole pipeline 

In [52]:
print(internet_search_result)
print(final_results )
print(Conclusion_Result)
print(COT_prepare_Dynamic_Rating)
print(dynamic_Rating)

total_sub_managers=6 sub_managers=[SubManagerQuery(manager_id='A', query="Analyze impact of potential Fed rate cuts on mortgage rates and transaction volume recovery for OPEN's iBuying business model", dimension='Market Conditions', priority=1, expected_impact='Positive'), SubManagerQuery(manager_id='B', query='Evaluate margin sustainability from 5-6% levels given pricing algorithm improvements and inventory holding period optimization', dimension='Financial', priority=2, expected_impact='Mixed'), SubManagerQuery(manager_id='C', query='Assess capital intensity risks and balance sheet capacity requirements for inventory holding during slow transaction periods', dimension='Financial Risk', priority=3, expected_impact='Negative'), SubManagerQuery(manager_id='D', query='Examine platform pivot success with agent partnerships in 11 markets and asset-light revenue stream expansion potential', dimension='Strategic', priority=4, expected_impact='Positive'), SubManagerQuery(manager_id='E', query

## Section 4). Final Result in Frontend

### Frontend Print This Code

In [35]:
# Q&Q.AI Report-Style Dashboard - Large Logo Design
import webbrowser
import os
from datetime import datetime

def visualize_qq_ai_report():
    """
    Report-style visualization: Chains first, then summary paragraphs
    Single horizontal bar with two segments for rewards
    Added color legend with language support
    Two-column layout for sell/buy side analysis
    Updated with Dynamic Rating structure + Price In/Price Out sections
    Higher scores = Green, Lower scores = Red
    Large logo design with invisible background
    """
    
    # Process final_results to get chain data - FIXED STRUCTURE
    chain_data = []
    
    for i, result_item in enumerate(final_results):
        if result_item.get('status') == 'success':
            # FIXED: Access the nested structure correctly
            result_data = result_item.get('result', {})
            chain_info = result_data.get('chain_of_thought', {})
            
            # Get the nested chain_of_thought data
            nested_chain = chain_info.get('chain_of_thought', {})
            events = nested_chain.get('events', [])
            
            # Extract starting event for title
            starting_event = events[0] if events else "Unknown starting event"
            
            chain_data.append({
                'index': i + 1,
                'query_key': result_item.get('query_key', f'Query {i+1}'),
                'direction': nested_chain.get('final_direction', 'Unknown'),
                'chain': nested_chain.get('impact_chain', 'Unknown'),
                'mermaid_code': chain_info.get('mermaid_code', ''),
                'starting_event': starting_event
            })
    
    print(f"✅ Extracted {len(chain_data)} chains from final_results")
    
    # Get data from internet_search_result - SEPARATE SELL/BUY SIDES
    try:
        sell_side_data = internet_search_result.sell_side_summary
        buy_side_data = internet_search_result.buy_side_summary
        business_logic_data = internet_search_result.business_logic
        print(f"✅ Successfully extracted from internet_search_result:")
        print(f"   - Sell side items: {len(sell_side_data)}")
        print(f"   - Buy side items: {len(buy_side_data)}")
        print(f"   - Business logic items: {len(business_logic_data)}")
    except NameError as e:
        print(f"⚠️ internet_search_result not found: {e}")
        sell_side_data = []
        buy_side_data = []
        business_logic_data = "No business logic available"
    
    # Get conclusion results
    try:
        catalyst = Conclusion_Result.get('Catalyst', 'Not available')
        short_term_impact = Conclusion_Result.get('short_term_impact', 'Not available')
        long_term_outlook = Conclusion_Result.get('long_term_outlook', 'Not available')
    except NameError:
        catalyst = 'Not available'
        short_term_impact = 'Not available'
        long_term_outlook = 'Not available'
    
    # Get Dynamic Rating scores AND Price In/Price Out data
    try:
        # Parse dynamic_rating variable (assuming it's a dict with the four scores + Price In/Out)
        if 'dynamic_Rating' in globals():
            dynamic_rating_data = dynamic_Rating
            short_term_risk = dynamic_rating_data.get('ShortTermRisk', 0.5)
            short_term_reward = dynamic_rating_data.get('ShortTermReward', 0.5)
            long_term_risk = dynamic_rating_data.get('LongTermRisk', 0.5)
            long_term_reward = dynamic_rating_data.get('LongTermReward', 0.5)
            price_in = dynamic_rating_data.get('Price_In', 'Price In analysis not available')
            price_out = dynamic_rating_data.get('Price_Out', 'Price Out analysis not available')
            dynamic_reasoning = dynamic_rating_data.get('Reasoning', 'No reasoning available')
            
            # Ensure they are floats
            short_term_risk = float(short_term_risk)
            short_term_reward = float(short_term_reward)
            long_term_risk = float(long_term_risk)
            long_term_reward = float(long_term_reward)
            
            print(f"✅ Dynamic Rating scores extracted:")
            print(f"   - Short Term Risk: {short_term_risk}")
            print(f"   - Short Term Reward: {short_term_reward}")
            print(f"   - Long Term Risk: {long_term_risk}")
            print(f"   - Long Term Reward: {long_term_reward}")
            print(f"✅ Price In/Price Out data extracted:")
            print(f"   - Price In: {price_in[:100]}...")
            print(f"   - Price Out: {price_out[:100]}...")
            
        else:
            raise ValueError("dynamic_Rating variable not found")
            
    except (KeyError, ValueError, TypeError) as e:
        print(f"❌ Failed to extract Dynamic Rating data: {e}")
        # Use default values
        short_term_risk = 0.5
        short_term_reward = 0.5
        long_term_risk = 0.5
        long_term_reward = 0.5
        price_in = "Price In analysis not available"
        price_out = "Price Out analysis not available"
        dynamic_reasoning = "Dynamic Rating analysis not available"
    
    # Convert lists to HTML paragraphs
    def list_to_paragraphs(items):
        if isinstance(items, list):
            return "".join([f"<p>{item}</p>\n" for item in items])
        else:
            return f"<p>{items}</p>\n"
    
    sell_side_html = list_to_paragraphs(sell_side_data)
    buy_side_html = list_to_paragraphs(buy_side_data)
    
    # Generate chain cards HTML
    chain_cards_html = ""
    for chain in chain_data:
        # Determine direction class
        direction_class = "unknown"
        if "Short Term Up" in chain['direction']:
            direction_class = "short-term-up"
        elif "Short Term Down" in chain['direction']:
            direction_class = "short-term-down"
        elif "Long Term Up" in chain['direction']:
            direction_class = "long-term-up"
        elif "Long Term Down" in chain['direction']:
            direction_class = "long-term-down"
        
        chain_cards_html += f"""
            <div class="chain-card">
                <div class="chain-header">
                    <div class="chain-title-wrapper">
                        <div class="chain-number">{chain['index']}</div>
                        <div class="chain-title">{chain['starting_event']}</div>
                    </div>
                    <span class="direction-badge {direction_class}">{chain['direction']}</span>
                </div>
                <div class="chain-content">{chain['chain']}</div>
                <div class="mermaid-wrapper">
                    <div class="mermaid">{chain['mermaid_code']}</div>
                </div>
            </div>
        """
    
    # NEW: Color scheme - Higher scores = Green, Lower scores = Red
    def get_score_class(score, is_risk=False):
        """
        For Risk: Lower is better (green), Higher is worse (red)
        For Reward: Higher is better (green), Lower is worse (red)
        """
        if is_risk:
            # For risk: lower scores are better (green)
            if score <= 0.3:
                return "excellent"  # Very low risk = green
            elif score <= 0.5:
                return "good"       # Low risk = light green
            elif score <= 0.7:
                return "warning"    # Medium risk = yellow
            else:
                return "danger"     # High risk = red
        else:
            # For reward: higher scores are better (green)
            if score >= 0.7:
                return "excellent"  # High reward = green
            elif score >= 0.5:
                return "good"       # Medium reward = light green
            elif score >= 0.3:
                return "warning"    # Low reward = yellow
            else:
                return "danger"     # Very low reward = red
    
    short_term_risk_class = get_score_class(short_term_risk, is_risk=True)
    long_term_risk_class = get_score_class(long_term_risk, is_risk=True)
    short_term_reward_class = get_score_class(short_term_reward, is_risk=False)
    long_term_reward_class = get_score_class(long_term_reward, is_risk=False)
    
    # Get ticker and other missing variables
    try:
        ticker_symbol = ticker
    except NameError:
        ticker_symbol = "UNKNOWN"
    
    # Generate dates
    analysis_date = datetime.now().strftime("%Y-%m-%d")
    report_date_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    chain_count = len(chain_data)
    
    # Language detection and text generation
    try:
        global_language = globals()['language']
        is_chinese = global_language.lower() == 'chinese'
    except KeyError:
        is_chinese = False
    
    # Generate language-specific text - UPDATED TITLES
    if is_chinese:
        color_legend_title = "影响链方向说明"
        color_legend_subtitle = "不同颜色代表不同的投资方向和时间维度"
        long_term_up_text = "长期上涨"
        long_term_down_text = "长期下跌"
        short_term_up_text = "短期上涨"
        short_term_down_text = "短期下跌"
        impact_chains_title = "定性AI"  # NEW TITLE
        sell_buy_title = "买卖分析总结"
        sell_side_title = "卖出分析"
        buy_side_title = "买入分析"
        business_logic_title = "商业逻辑总结"
        conclusion_title = "定量AI"  # NEW TITLE
        short_term_risk_text = "短期风险评分"
        long_term_risk_text = "长期风险评分"
        short_term_reward_text = "短期回报评分"
        long_term_reward_text = "长期回报评分"
        price_in_title = "当前价格已反映"  # NEW TITLE
        price_out_title = "当前价格未反映"  # NEW TITLE
        key_catalysts_text = "关键催化剂"
        short_term_impact_text = "短期影响"
        long_term_outlook_text = "长期展望"
        dynamic_reasoning_title = "AI风险评级分析说明"
        footer_text = "由 Q&Q.AI  - 定量&定性 AI"
        report_generated_text = "报告生成时间"
        copyright_text = "© 2025 Q&Q.AI - 连接数据智能"
        logo_description = "定量与定性AI投资分析系统"
    else:
        color_legend_title = "Impact Chain Direction Legend"
        color_legend_subtitle = "Different colors represent different investment directions and time horizons"
        long_term_up_text = "Long Term Up"
        long_term_down_text = "Long Term Down"
        short_term_up_text = "Short Term Up"
        short_term_down_text = "Short Term Down"
        impact_chains_title = "Qualitative AI"  # NEW TITLE
        sell_buy_title = "Sell & Buy Analysis Summary"
        sell_side_title = "Sell Side Analysis"
        buy_side_title = "Buy Side Analysis"
        business_logic_title = "Business Logic Summary"
        conclusion_title = "Quantitative AI"  # NEW TITLE
        short_term_risk_text = "Short Term Risk Score"
        long_term_risk_text = "Long Term Risk Score"
        short_term_reward_text = "Short Term Reward Score"
        long_term_reward_text = "Long Term Reward Score"
        price_in_title = "Currently Priced In"  # NEW TITLE
        price_out_title = "Not Yet Priced In"  # NEW TITLE
        key_catalysts_text = "Key Catalysts"
        short_term_impact_text = "Short Term Impact"
        long_term_outlook_text = "Long Term Outlook"
        dynamic_reasoning_title = "Dynamic Rating Analysis Explanation"
        footer_text = "Generated by Q&Q.AI - Quantitative & Qualitative AI Investment Analysis System"
        report_generated_text = "Report generated on"
        copyright_text = "© 2025 Q&Q.AI - Bridging Data Intelligence"
        logo_description = "Quantitative & Qualitative AI Investment Analysis System"
    
    # Generate color legend HTML
    color_legend_html = f"""
        <div class="color-legend">
            <h3 class="legend-title">{color_legend_title}</h3>
            <p class="legend-subtitle">{color_legend_subtitle}</p>
            <div class="legend-grid">
                <div class="legend-item">
                    <div class="legend-color long-term-up"></div>
                    <span class="legend-text">{long_term_up_text}</span>
                </div>
                <div class="legend-item">
                    <div class="legend-color long-term-down"></div>
                    <span class="legend-text">{long_term_down_text}</span>
                </div>
                <div class="legend-item">
                    <div class="legend-color short-term-up"></div>
                    <span class="legend-text">{short_term_up_text}</span>
                </div>
                <div class="legend-item">
                    <div class="legend-color short-term-down"></div>
                    <span class="legend-text">{short_term_down_text}</span>
                </div>
            </div>
        </div>
    """
    
    # Generate section titles with rings
    def get_section_title_with_ring(title_text, is_quantitative=True):
        ring_class = "ring-quantitative" if is_quantitative else "ring-qualitative"
        ring_glow_class = "ring-glow-quantitative" if is_quantitative else "ring-glow-qualitative"
        
        return f"""
            <div class="section-title-with-ring">
                <div class="single-ring {ring_class}">
                    <div class="ring-glow {ring_glow_class}"></div>
                </div>
                <div class="section-title-text">{title_text}</div>
            </div>
        """
    
    # Generate section titles
    impact_chains_title_html = get_section_title_with_ring(impact_chains_title, is_quantitative=False)
    conclusion_title_html = get_section_title_with_ring(conclusion_title, is_quantitative=True)
    
    # Generate report-style HTML - LARGE LOGO DESIGN
    html_content = f"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Q&Q.AI - Impact Chain Analysis Results</title>
    <script src="https://cdn.jsdelivr.net/npm/mermaid/dist/mermaid.min.js"></script>
    <style>
        * {{
            margin: 0;
            padding: 0;
            box-sizing: border-box;
        }}
        
        body {{
            font-family: 'Courier New', 'Monaco', 'Menlo', monospace;
            background: linear-gradient(135deg, #0f0f23 0%, #1a1a2e 50%, #16213e 100%);
            color: #ffffff;
            min-height: 100vh;
            position: relative;
            overflow-x: hidden;
        }}

        .main-container {{
            position: relative;
            z-index: 2;
            max-width: 1400px;
            margin: 0 auto;
            padding: 40px 20px;
        }}

        /* Large Logo Section - Invisible Background */
        .logo-section {{
            text-align: center;
            margin-bottom: 60px;
            padding: 60px 20px;
            /* NO BACKGROUND - Invisible chunk */
            background: transparent;
            border: none;
            border-radius: 0;
        }}

        .logo-svg {{
            width: 500px;
            height: 200px;
            filter: drop-shadow(0 0 40px rgba(102, 126, 234, 0.8));
            animation: pulse-glow 4s ease-in-out infinite;
        }}

        @keyframes pulse-glow {{
            0%, 100% {{ filter: drop-shadow(0 0 40px rgba(102, 126, 234, 0.8)); }}
            50% {{ filter: drop-shadow(0 0 60px rgba(118, 75, 162, 1)); }}
        }}

        .logo-text {{
            font-size: 48px;
            font-weight: 200;
            color: #ffffff;
            letter-spacing: 12px;
            margin-top: 30px;
            text-shadow: 0 0 30px rgba(102, 126, 234, 0.8);
        }}

        .logo-description {{
            font-size: 20px;
            font-weight: 100;
            color: #a0aec0;
            letter-spacing: 4px;
            margin-top: 15px;
            opacity: 0.9;
        }}

        .ticker-bar {{
            background: rgba(255, 255, 255, 0.02);
            backdrop-filter: blur(20px);
            border: 1px solid rgba(255, 255, 255, 0.1);
            border-radius: 20px;
            padding: 20px 40px;
            margin-bottom: 40px;
            display: flex;
            justify-content: space-between;
            align-items: center;
        }}

        .ticker-symbol {{
            font-size: 2em;
            font-weight: 600;
            color: #667eea;
            text-shadow: 0 0 20px rgba(102, 126, 234, 0.5);
        }}

        .section {{
            margin-bottom: 60px;
        }}

        /* Single Ring Design */
        .ring-container {{
            display: inline-flex;
            align-items: center;
            gap: 15px;
        }}

        .single-ring {{
            width: 25px;
            height: 25px;
            border: 2px solid;
            border-radius: 50%;
            position: relative;
            animation: ring-pulse 2s ease-in-out infinite;
        }}

        .ring-quantitative {{
            border-color: #667eea;
            box-shadow: 0 0 15px rgba(102, 126, 234, 0.6);
        }}

        .ring-qualitative {{
            border-color: #764ba2;
            box-shadow: 0 0 15px rgba(118, 75, 162, 0.6);
        }}

        @keyframes ring-pulse {{
            0%, 100% {{ 
                transform: scale(1);
                opacity: 0.8;
            }}
            50% {{ 
                transform: scale(1.1);
                opacity: 1;
            }}
        }}

        .ring-glow {{
            position: absolute;
            top: -2px;
            left: -2px;
            right: -2px;
            bottom: -2px;
            border-radius: 50%;
            opacity: 0.3;
            animation: ring-glow 3s ease-in-out infinite;
        }}

        .ring-glow-quantitative {{
            border: 1px solid #667eea;
            box-shadow: 0 0 20px rgba(102, 126, 234, 0.4);
        }}

        .ring-glow-qualitative {{
            border: 1px solid #764ba2;
            box-shadow: 0 0 20px rgba(118, 75, 162, 0.4);
        }}

        @keyframes ring-glow {{
            0%, 100% {{ opacity: 0.3; }}
            50% {{ opacity: 0.6; }}
        }}

        /* Section Titles with Rings */
        .section-title-with-ring {{
            display: flex;
            align-items: center;
            gap: 15px;
            font-size: 2em;
            font-weight: 300;
            letter-spacing: 3px;
            margin-bottom: 30px;
            padding-bottom: 15px;
            border-bottom: 2px solid rgba(102, 126, 234, 0.3);
            text-transform: uppercase;
            font-family: 'Courier New', 'Monaco', 'Menlo', monospace;
        }}

        .section-title-text {{
            background: linear-gradient(45deg, #667eea, #764ba2);
            -webkit-background-clip: text;
            -webkit-text-fill-color: transparent;
            background-clip: text;
        }}

        .section-title {{
            font-size: 2em;
            font-weight: 300;
            letter-spacing: 3px;
            margin-bottom: 30px;
            padding-bottom: 15px;
            border-bottom: 2px solid rgba(102, 126, 234, 0.3);
            text-transform: uppercase;
            font-family: 'Courier New', 'Monaco', 'Menlo', monospace;
        }}

        .color-legend {{
            background: rgba(255, 255, 255, 0.03);
            backdrop-filter: blur(10px);
            border: 1px solid rgba(255, 255, 255, 0.1);
            border-radius: 20px;
            padding: 30px;
            margin-bottom: 40px;
        }}

        .legend-title {{
            font-size: 1.5em;
            font-weight: 400;
            margin-bottom: 10px;
            color: #667eea;
            text-align: center;
        }}

        .legend-subtitle {{
            font-size: 1em;
            color: #a0aec0;
            text-align: center;
            margin-bottom: 25px;
        }}

        .legend-grid {{
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(200px, 1fr));
            gap: 20px;
        }}

        .legend-item {{
            display: flex;
            align-items: center;
            gap: 15px;
            padding: 15px;
            background: rgba(0, 0, 0, 0.2);
            border-radius: 10px;
        }}

        .legend-color {{
            width: 30px;
            height: 30px;
            border-radius: 50%;
            flex-shrink: 0;
        }}

        .legend-color.long-term-up {{
            background: linear-gradient(135deg, #3498db, #5dade2);
        }}

        .legend-color.long-term-down {{
            background: linear-gradient(135deg, #e67e22, #f39c12);
        }}

        .legend-color.short-term-up {{
            background: linear-gradient(135deg, #27ae60, #2ecc71);
        }}

        .legend-color.short-term-down {{
            background: linear-gradient(135deg, #e74c3c, #c0392b);
        }}

        .legend-text {{
            font-weight: 500;
            font-size: 1.1em;
        }}

        .chain-card {{
            background: rgba(255, 255, 255, 0.03);
            backdrop-filter: blur(10px);
            border: 1px solid rgba(255, 255, 255, 0.1);
            border-radius: 20px;
            padding: 30px;
            margin-bottom: 30px;
            transition: all 0.3s ease;
        }}

        .chain-header {{
            display: flex;
            justify-content: space-between;
            align-items: flex-start;
            margin-bottom: 20px;
        }}

        .chain-title-wrapper {{
            display: flex;
            align-items: center;
            gap: 15px;
        }}

        .chain-number {{
            background: linear-gradient(135deg, #667eea, #764ba2);
            width: 40px;
            height: 40px;
            border-radius: 50%;
            display: flex;
            align-items: center;
            justify-content: center;
            font-weight: 600;
            font-size: 1.2em;
            flex-shrink: 0;
        }}

        .chain-title {{
            font-size: 1.1em;
            font-weight: 400;
            color: #ffffff;
            line-height: 1.4;
        }}

        .direction-badge {{
            padding: 8px 20px;
            border-radius: 30px;
            font-weight: 500;
            font-size: 0.85em;
            text-transform: uppercase;
            letter-spacing: 1px;
            white-space: nowrap;
        }}

        .direction-badge.short-term-up {{
            background: rgba(39, 174, 96, 0.2);
            color: #27ae60;
            border: 1px solid #27ae60;
        }}

        .direction-badge.short-term-down {{
            background: rgba(231, 76, 60, 0.2);
            color: #e74c3c;
            border: 1px solid #e74c3c;
        }}

        .direction-badge.long-term-up {{
            background: rgba(52, 152, 219, 0.2);
            color: #3498db;
            border: 1px solid #3498db;
        }}

        .direction-badge.long-term-down {{
            background: rgba(230, 126, 34, 0.2);
            color: #e67e22;
            border: 1px solid #e67e22;
        }}

        .chain-content {{
            background: rgba(0, 0, 0, 0.2);
            border-radius: 10px;
            padding: 20px;
            margin-bottom: 20px;
            font-family: 'Courier New', monospace;
            font-size: 0.95em;
            line-height: 1.6;
            color: #cbd5e0;
        }}

        .mermaid-wrapper {{
            background: rgba(255, 255, 255, 0.05);
            border-radius: 10px;
            padding: 20px;
            overflow-x: auto;
        }}

        .summary-card {{
            background: rgba(255, 255, 255, 0.03);
            backdrop-filter: blur(10px);
            border: 1px solid rgba(255, 255, 255, 0.1);
            border-radius: 20px;
            padding: 40px;
            margin-bottom: 30px;
        }}

        .summary-content {{
            font-size: 1.1em;
            line-height: 1.8;
            color: #cbd5e0;
        }}

        .summary-content p {{
            margin-bottom: 15px;
        }}

        /* TWO COLUMN LAYOUT FOR SELL/BUY */
        .sell-buy-container {{
            display: grid;
            grid-template-columns: 1fr 1fr;
            gap: 30px;
            margin-bottom: 30px;
        }}

        .sell-side-card {{
            background: rgba(231, 76, 60, 0.05);
            border: 1px solid rgba(231, 76, 60, 0.2);
            border-radius: 20px;
            padding: 30px;
        }}

        .buy-side-card {{
            background: rgba(39, 174, 96, 0.05);
            border: 1px solid rgba(39, 174, 96, 0.2);
            border-radius: 20px;
            padding: 30px;
        }}

        .side-title {{
            font-size: 1.3em;
            font-weight: 600;
            margin-bottom: 20px;
            text-align: center;
            text-transform: uppercase;
            letter-spacing: 2px;
        }}

        .sell-side-title {{
            color: #e74c3c;
        }}

        .buy-side-title {{
            color: #27ae60;
        }}

        .side-content {{
            font-size: 1em;
            line-height: 1.7;
            color: #cbd5e0;
        }}

        .side-content p {{
            margin-bottom: 12px;
        }}

        /* NEW: FOUR PANEL LAYOUT FOR RISK/REWARD */
        .risk-reward-grid {{
            display: grid;
            grid-template-columns: 1fr 1fr;
            gap: 30px;
            margin-bottom: 40px;
        }}

        .risk-panel {{
            display: grid;
            grid-template-columns: 1fr 1fr;
            gap: 20px;
        }}

        .reward-panel {{
            display: grid;
            grid-template-columns: 1fr 1fr;
            gap: 20px;
        }}

        .metric-card {{
            background: rgba(255, 255, 255, 0.03);
            backdrop-filter: blur(10px);
            border: 1px solid rgba(255, 255, 255, 0.1);
            border-radius: 20px;
            padding: 30px;
            text-align: center;
            transition: all 0.3s ease;
        }}

        .metric-value {{
            font-size: 3em;
            font-weight: 600;
            margin-bottom: 10px;
        }}

        /* NEW COLOR SCHEME: Higher scores = Green, Lower scores = Red */
        .metric-value.excellent {{
            color: #27ae60;
            text-shadow: 0 0 20px rgba(39, 174, 96, 0.5);
        }}

        .metric-value.good {{
            color: #2ecc71;
            text-shadow: 0 0 20px rgba(46, 204, 113, 0.5);
        }}

        .metric-value.warning {{
            color: #f39c12;
            text-shadow: 0 0 20px rgba(243, 156, 18, 0.5);
        }}

        .metric-value.danger {{
            color: #e74c3c;
            text-shadow: 0 0 20px rgba(231, 76, 60, 0.5);
        }}

        .metric-label {{
            font-size: 0.9em;
            color: #a0aec0;
            text-transform: uppercase;
            letter-spacing: 1px;
        }}

        /* NEW: TWO COLUMN LAYOUT FOR PRICE IN/PRICE OUT */
        .price-analysis-container {{
            display: grid;
            grid-template-columns: 1fr 1fr;
            gap: 30px;
            margin-bottom: 40px;
        }}

        .price-in-card {{
            background: rgba(39, 174, 96, 0.05);
            border: 1px solid rgba(39, 174, 96, 0.2);
            border-radius: 20px;
            padding: 30px;
        }}

        .price-out-card {{
            background: rgba(231, 76, 60, 0.05);
            border: 1px solid rgba(231, 76, 60, 0.2);
            border-radius: 20px;
            padding: 30px;
        }}

        .price-title {{
            font-size: 1.3em;
            font-weight: 600;
            margin-bottom: 20px;
            text-align: center;
            text-transform: uppercase;
            letter-spacing: 2px;
        }}

        .price-in-title {{
            color: #27ae60;
        }}

        .price-out-title {{
            color: #e74c3c;
        }}

        .price-content {{
            font-size: 1em;
            line-height: 1.7;
            color: #cbd5e0;
        }}

        /* NEW: Dynamic Reasoning Bar */
        .reasoning-container {{
            background: rgba(102, 126, 234, 0.1);
            border: 1px solid rgba(102, 126, 234, 0.3);
            border-radius: 20px;
            padding: 30px;
            margin-bottom: 40px;
        }}

        .reasoning-title {{
            font-size: 1.3em;
            margin-bottom: 20px;
            color: #667eea;
            font-weight: 500;
            text-align: center;
        }}

        .reasoning-bar {{
            background: rgba(0, 0, 0, 0.3);
            border-radius: 15px;
            padding: 25px;
            border-left: 4px solid #667eea;
        }}

        .reasoning-text {{
            font-size: 1.1em;
            line-height: 1.7;
            color: #cbd5e0;
            text-align: justify;
        }}

        .conclusion-card {{
            background: rgba(102, 126, 234, 0.1);
            border: 1px solid rgba(102, 126, 234, 0.3);
            border-radius: 20px;
            padding: 30px;
            margin-bottom: 20px;
        }}

        .conclusion-title {{
            font-size: 1.3em;
            margin-bottom: 15px;
            color: #667eea;
            font-weight: 500;
        }}

        .conclusion-text {{
            line-height: 1.7;
            color: #cbd5e0;
        }}

        .footer {{
            text-align: center;
            padding: 40px 20px;
            margin-top: 80px;
            border-top: 1px solid rgba(255, 255, 255, 0.1);
            color: #a0aec0;
            font-size: 0.9em;
        }}

        /* Responsive design */
        @media (max-width: 768px) {{
            .sell-buy-container {{
                grid-template-columns: 1fr;
            }}
            .risk-reward-grid {{
                grid-template-columns: 1fr;
            }}
            .risk-panel, .reward-panel {{
                grid-template-columns: 1fr;
            }}
            .price-analysis-container {{
                grid-template-columns: 1fr;
            }}
        }}
    </style>
</head>
<body>
    <div class="main-container">
        <!-- Large Logo Section - Invisible Background -->
        <div class="logo-section">
            <svg class="logo-svg" viewBox="0 0 360 150" xmlns="http://www.w3.org/2000/svg">
                <defs>
                    <linearGradient id="cleanGradient" x1="0%" y1="0%" x2="100%" y2="100%">
                        <stop offset="0%" style="stop-color:#667eea;stop-opacity:1">
                            <animate attributeName="stop-color" 
                                    values="#667eea;#764ba2;#667eea" 
                                    dur="4s" repeatCount="indefinite"/>
                        </stop>
                        <stop offset="100%" style="stop-color:#764ba2;stop-opacity:1">
                            <animate attributeName="stop-color" 
                                    values="#764ba2;#667eea;#764ba2" 
                                    dur="4s" repeatCount="indefinite"/>
                        </stop>
                    </linearGradient>
                    <filter id="subtleGlow" x="-20%" y="-20%" width="140%" height="140%">
                        <feGaussianBlur stdDeviation="3" result="coloredBlur"/>
                        <feMerge> 
                            <feMergeNode in="coloredBlur"/>
                            <feMergeNode in="SourceGraphic"/>
                        </feMerge>
                    </filter>
                </defs>
                
                <!-- First Ellipse (Quantitative) -->
                <ellipse cx="150" cy="65" rx="50" ry="25" 
                      fill="none"
                      stroke="url(#cleanGradient)" 
                      stroke-width="5"
                      filter="url(#subtleGlow)"
                      opacity="0.9">
                    <animate attributeName="opacity" values="0.9;1;0.9" dur="3s" repeatCount="indefinite"/>
                </ellipse>

                <!-- Second Ellipse (Qualitative) -->
                <ellipse cx="210" cy="85" rx="50" ry="25" 
                      fill="none"
                      stroke="url(#cleanGradient)" 
                      stroke-width="5"
                      filter="url(#subtleGlow)"
                      opacity="0.9">
                    <animate attributeName="opacity" values="0.9;1;0.9" dur="3s" repeatCount="indefinite" begin="1.5s"/>
                </ellipse>

                <!-- Intersection area highlight -->
                <ellipse cx="180" cy="75" rx="20" ry="12" 
                      fill="url(#cleanGradient)" 
                      opacity="0.25"
                      filter="url(#subtleGlow)">
                    <animate attributeName="opacity" values="0.25;0.4;0.25" dur="3s" repeatCount="indefinite"/>
                </ellipse>
            </svg>
            <div class="logo-text">Q&Q.AI</div>
            <div class="logo-description">{logo_description}</div>
        </div>

        <!-- Ticker info bar -->
        <div class="ticker-bar">
            <div class="ticker-symbol">{ticker_symbol}</div>
            <div>Analysis Date: {analysis_date}</div>
            <div>{chain_count} Impact Chains Analyzed</div>
        </div>

        <!-- Color Legend -->
        {color_legend_html}

        <!-- Impact Chains Section - NEW TITLE WITH RING -->
        <div class="section">
            {impact_chains_title_html}
            {chain_cards_html}
        </div>

        <!-- Sell & Buy Analysis - TWO COLUMN LAYOUT -->
        <div class="section">
            <h2 class="section-title">{sell_buy_title}</h2>
            <div class="sell-buy-container">
                <div class="sell-side-card">
                    <h3 class="side-title sell-side-title">{sell_side_title}</h3>
                    <div class="side-content">
                        {sell_side_html}
                    </div>
                </div>
                <div class="buy-side-card">
                    <h3 class="side-title buy-side-title">{buy_side_title}</h3>
                    <div class="side-content">
                        {buy_side_html}
                    </div>
                </div>
            </div>
        </div>

        <!-- Business Logic Summary -->
        <div class="section">
            <h2 class="section-title">{business_logic_title}</h2>
            <div class="summary-card">
                <div class="summary-content">
                    <p>{business_logic_data}</p>
                </div>
            </div>
        </div>

        <!-- Conclusion Analysis - NEW TITLE WITH RING -->
        <div class="section">
            {conclusion_title_html}

            <!-- KEEP: Four Panel Risk/Reward Layout -->
            <div class="risk-reward-grid">
                <!-- Risk Panel -->
                <div class="risk-panel">
                    <div class="metric-card">
                        <div class="metric-value {short_term_risk_class}">{short_term_risk}/1.0</div>
                        <div class="metric-label">{short_term_risk_text}</div>
                    </div>
                    <div class="metric-card">
                        <div class="metric-value {long_term_risk_class}">{long_term_risk}/1.0</div>
                        <div class="metric-label">{long_term_risk_text}</div>
                    </div>
                </div>
                
                <!-- Reward Panel -->
                <div class="reward-panel">
                    <div class="metric-card">
                        <div class="metric-value {short_term_reward_class}">{short_term_reward}/1.0</div>
                        <div class="metric-label">{short_term_reward_text}</div>
                    </div>
                    <div class="metric-card">
                        <div class="metric-value {long_term_reward_class}">{long_term_reward}/1.0</div>
                        <div class="metric-label">{long_term_reward_text}</div>
                    </div>
                </div>
            </div>

            <!-- NEW: Two Column Price In/Price Out Layout -->
            <div class="price-analysis-container">
                <div class="price-in-card">
                    <h3 class="price-title price-in-title">{price_in_title}</h3>
                    <div class="price-content">
                        <p>{price_in}</p>
                    </div>
                </div>
                <div class="price-out-card">
                    <h3 class="price-title price-out-title">{price_out_title}</h3>
                    <div class="price-content">
                        <p>{price_out}</p>
                    </div>
                </div>
            </div>

            <!-- NEW: Dynamic Reasoning Bar -->
            <div class="reasoning-container">
                <h3 class="reasoning-title">{dynamic_reasoning_title}</h3>
                <div class="reasoning-bar">
                    <div class="reasoning-text">{dynamic_reasoning}</div>
                </div>
            </div>

            <!-- Key Catalysts -->
            <div class="conclusion-card">
                <h3 class="conclusion-title">{key_catalysts_text}</h3>
                <div class="conclusion-text">{catalyst}</div>
            </div>

            <!-- Short Term Impact -->
            <div class="conclusion-card">
                <h3 class="conclusion-title">{short_term_impact_text}</h3>
                <div class="conclusion-text">{short_term_impact}</div>
            </div>

            <!-- Long Term Outlook -->
            <div class="conclusion-card">
                <h3 class="conclusion-title">{long_term_outlook_text}</h3>
                <div class="conclusion-text">{long_term_outlook}</div>
            </div>
        </div>

        <!-- Footer -->
        <div class="footer">
            <p>{footer_text}</p>
            <p>{report_generated_text}: {report_date_time}</p>
            <p>{copyright_text}</p>
        </div>
    </div>

    <script>
        // Initialize Mermaid
        mermaid.initialize({{
            startOnLoad: true,
            theme: 'dark',
            themeVariables: {{
                primaryColor: '#667eea',
                primaryTextColor: '#fff',
                primaryBorderColor: '#764ba2',
                lineColor: '#667eea',
                secondaryColor: '#764ba2',
                background: 'rgba(255, 255, 255, 0.05)',
                mainBkg: 'rgba(255, 255, 255, 0.05)',
                secondBkg: 'rgba(102, 126, 234, 0.1)',
                fontFamily: 'Segoe UI',
                fontSize: '14px'
            }},
            flowchart: {{
                useMaxWidth: true,
                htmlLabels: true,
                curve: 'basis'
            }}
        }});
    </script>
</body>
</html>"""
    
    # Save to file
    filename = f"qq_ai_report_{ticker_symbol}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.html"
    with open(filename, 'w', encoding='utf-8') as f:
        f.write(html_content)
    
    # Open in browser
    webbrowser.open(f"file://{os.path.abspath(filename)}")
visualize_qq_ai_report()


✅ Extracted 6 chains from final_results
✅ Successfully extracted from internet_search_result:
   - Sell side items: 5
   - Buy side items: 5
   - Business logic items: 5
✅ Dynamic Rating scores extracted:
   - Short Term Risk: 0.75
   - Short Term Reward: 0.35
   - Long Term Risk: 0.45
   - Long Term Reward: 0.65
✅ Price In/Price Out data extracted:
   - Price In: 当前价格已充分反映新CEO任命和创始人回归的积极消息（单日暴涨68.69%），美联储降息预期和Q2盈利改善也已定价。58%的近期涨幅显示市场对短期利好反应过度，反向股票拆分风险部分定价但未完全消化。...
   - Price Out: 市场低估了平台化转型的结构性价值（毛利率从负转正至5-6%），轻资产模式带来的现金流改善（自由现金流收益率2.11%），以及运营效率提升33%的长期影响。同时高估了短期降息对业务的即时提振效果。...


# --------------------------------------- Above is Supervisor to Frontend Logic --------------------------------

### Update Logic For the whole Pipeline 

In [263]:
from tavily import TavilyClient

In [264]:

def update_check(ticker):
    """
    Check the most updated news for a ticker with detailed bullet point summary
    
    Args:
        ticker (str): Stock ticker symbol (e.g., 'MSTR', 'NVDA')
    
    Returns:
        str: Summarized sources with bullet points
    """
    try:
        # Initialize Tavily client
        client = TavilyClient("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")
        
        print(f"🔍 Checking latest updates for {ticker}...")
        
        # Search for most recent news and updates
        query = f"Latest news and updates for {ticker} stock - recent developments, earnings, announcements, analyst ratings, price movements. Provide detailed bullet point summary of key information."
        
        response = client.search(
            query=query,
            include_answer="advanced",
            search_depth="advanced",
            topic="finance",
            max_results=15
        )
        
        # Extract content from results
        content_list = []
        if 'results' in response:
            for result in response['results']:
                if 'content' in result and result['content']:
                    content_list.append(result['content'])
        
        # Get the answer summary
        answer = response.get('answer', '')
        
        # Create summarized sources
        summarized_sources = f"""
📰 LATEST UPDATES FOR {ticker.upper()}:
{answer}

📊 DETAILED SOURCES:
"""
        
        # Add bullet points from each source
        for i, content in enumerate(content_list[:10]):  # Limit to top 10 sources
            summarized_sources += f"• Source {i+1}: {content[:200]}...\n"
        
        print(f"✅ Update check complete for {ticker}")
        print(f"📈 Sources found: {len(response.get('results', []))}")
        print(f"⏱️ Response time: {response.get('response_time', 0)} seconds")
        
        return summarized_sources
        
    except Exception as e:
        print(f"❌ Error in update check for {ticker}: {e}")
        return f"❌ Error occurred during update check for {ticker}: {e}"

# Usage example:
# latest_updates = update_check("MSTR")
# print(latest_updates)